# View and analyze the Modal prediction ledger

This notebook calls the deployed `read_prediction_ledger` Modal function. It does not read `.env` and does not need the Explaining Markets API key. Your local Modal CLI must be authenticated, and the reader function must already be deployed.

In [1]:
from collections import Counter, defaultdict
from html import escape
from statistics import mean

import modal
from IPython.display import HTML, display

c:\Users\wfpin\Desktop\Projects\explaining-markets\.venv\Lib\site-packages\modal\_utils\async_utils.py:45: DeprecationWarning: 'asyncio.WindowsSelectorEventLoopPolicy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())
c:\Users\wfpin\Desktop\Projects\explaining-markets\.venv\Lib\site-packages\modal\_utils\async_utils.py:45: DeprecationWarning: 'asyncio.set_event_loop_policy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())


## 1. Read the deployed ledger

`APP_NAME` and `FUNCTION_NAME` must match the deployed Modal app and function. The current reader uses `limit=0` to mean all rows.

In [2]:
APP_NAME = "explaining-markets-starter"
FUNCTION_NAME = "read_prediction_ledger"
LIMIT = None  # Current deployed function interprets None as all rows.

read_ledger = modal.Function.from_name(APP_NAME, FUNCTION_NAME)
rows = read_ledger.remote(LIMIT)

if not isinstance(rows, list):
    raise TypeError(f"Expected a list from Modal, got {type(rows).__name__}")

print(f"Retrieved {len(rows)} ledger rows from {APP_NAME}.")

Retrieved 644 ledger rows from explaining-markets-starter.


## 2. Validate the ledger schema

In [3]:
EXPECTED_COLUMNS = {
    "event_id",
    "ticker",
    "prompt_version",
    "predicted_percentile",
    "confidence",
    "rules_applied",
    "realized_abnormal",
    "realized_percentile",
}

schema_issues = []
for index, row in enumerate(rows):
    if not isinstance(row, dict):
        schema_issues.append(f"row {index}: expected dict, got {type(row).__name__}")
        continue
    missing = EXPECTED_COLUMNS - set(row)
    if missing:
        schema_issues.append(f"row {index}: missing {sorted(missing)}")
    percentile = row.get("predicted_percentile")
    if percentile is not None and not 0 <= float(percentile) <= 1:
        schema_issues.append(f"row {index}: percentile outside [0, 1]: {percentile}")
    if not isinstance(row.get("rules_applied", []), list):
        schema_issues.append(f"row {index}: rules_applied is not a list")

if schema_issues:
    print(f"Found {len(schema_issues)} schema issues:")
    for issue in schema_issues[:20]:
        print(" -", issue)
else:
    print("All ledger rows match the expected schema.")

All ledger rows match the expected schema.


## 3. Display prediction rows

In [4]:
def display_table(table_rows, columns=None, limit=50):
    table_rows = list(table_rows)[:limit]
    if not table_rows:
        print("No rows to display.")
        return
    columns = columns or list(table_rows[0])
    header = "".join(f"<th>{escape(str(column))}</th>" for column in columns)
    body = []
    for row in table_rows:
        cells = "".join(
            f"<td>{escape(str(row.get(column, '')))}</td>"
            for column in columns
        )
        body.append(f"<tr>{cells}</tr>")
    display(HTML(
        "<div style='overflow:auto;max-height:600px'>"
        f"<table><thead><tr>{header}</tr></thead>"
        f"<tbody>{''.join(body)}</tbody></table></div>"
    ))

prediction_columns = [
    "event_id", "ticker", "prompt_version",
    "predicted_percentile", "confidence", "rules_applied",
    "realized_abnormal", "realized_percentile",
]
display_table(rows, prediction_columns)

event_id,ticker,prompt_version,predicted_percentile,confidence,rules_applied,realized_abnormal,realized_percentile
ea_LNG_Q2_2026,LNG,0.1.0,0.95,high,"['GLB-EXPECT-03', 'Q3-CAL-01', 'Q3-CAL-03']",None,None
ea_CTMX_Q2_2026,CTMX,0.1.0,0.9,high,"['GLB-EXPECT-03', 'Q3-CAL-01']",None,None
ea_EZPW_Q3_2026,EZPW,0.1.0,0.25,medium,"['P5', 'GLB-EXPECT-01', 'GLB-TONE-01', 'Q3-CAL-02']",None,None
ea_ASTE_Q2_2026,ASTE,0.1.0,0.15,high,"['GLB-GUID-01', 'Q3-CAL-01', 'Q3-CAL-02']",None,None
ea_LASR_Q2_2026,LASR,0.1.0,0.1,medium,"['GLB-GUID-01', 'Q3-CAL-01', 'Q3-CAL-02']",None,None
ea_BSY_Q2_2026,BSY,0.1.0,0.15,medium,"['GLB-GUID-01', 'GLB-TONE-01', 'GLB-EXPECT-02', 'Q3-CAL-01', 'Q3-CAL-02']",None,None
ea_SWX_Q2_2026,SWX,0.1.0,0.12,medium,"['GLB-QUAL-01', 'GLB-GUID-01', 'Q3-CAL-01', 'Q3-CAL-02']",None,None
ea_EHTH_Q2_2026,EHTH,0.1.0,0.08,medium,"['GLB-GUID-02', 'Q3-CAL-01', 'Q3-CAL-02']",None,None
ea_VNDA_Q2_2026,VNDA,0.1.0,0.08,high,"['GLB-GUID-01', 'GLB-QUAL-03', 'Q3-CAL-01', 'Q3-CAL-02']",None,None
ea_PSX_Q2_2026,PSX,0.1.0,0.82,medium,"['GLB-QUAL-01', 'GLB-CAP-01', 'Q3-CAL-01', 'Q3-CAL-03']",None,None


## 4. Expand `rules_applied`

Each output row below represents one prediction/rule pair. A prediction influenced by three rules therefore produces three rows.

In [5]:
rule_rows = []
for row in rows:
    for rule_id in row.get("rules_applied") or []:
        rule_rows.append({
            "event_id": row.get("event_id"),
            "ticker": row.get("ticker"),
            "prompt_version": row.get("prompt_version"),
            "rule_id": rule_id,
            "predicted_percentile": row.get("predicted_percentile"),
            "confidence": row.get("confidence"),
            "realized_abnormal": row.get("realized_abnormal"),
            "realized_percentile": row.get("realized_percentile"),
        })

empty_rule_rows = sum(not (row.get("rules_applied") or []) for row in rows)
print(f"Expanded {len(rule_rows)} prediction/rule pairs.")
print(f"Predictions with no reported rules: {empty_rule_rows}")
display_table(rule_rows)

Expanded 2739 prediction/rule pairs.
Predictions with no reported rules: 0


event_id,ticker,prompt_version,rule_id,predicted_percentile,confidence,realized_abnormal,realized_percentile
ea_LNG_Q2_2026,LNG,0.1.0,GLB-EXPECT-03,0.95,high,None,None
ea_LNG_Q2_2026,LNG,0.1.0,Q3-CAL-01,0.95,high,None,None
ea_LNG_Q2_2026,LNG,0.1.0,Q3-CAL-03,0.95,high,None,None
ea_CTMX_Q2_2026,CTMX,0.1.0,GLB-EXPECT-03,0.9,high,None,None
ea_CTMX_Q2_2026,CTMX,0.1.0,Q3-CAL-01,0.9,high,None,None
ea_EZPW_Q3_2026,EZPW,0.1.0,P5,0.25,medium,None,None
ea_EZPW_Q3_2026,EZPW,0.1.0,GLB-EXPECT-01,0.25,medium,None,None
ea_EZPW_Q3_2026,EZPW,0.1.0,GLB-TONE-01,0.25,medium,None,None
ea_EZPW_Q3_2026,EZPW,0.1.0,Q3-CAL-02,0.25,medium,None,None
ea_ASTE_Q2_2026,ASTE,0.1.0,GLB-GUID-01,0.15,high,None,None


## 5. Rule usage summary

This shows how often each rule fired and the average prediction when it fired. It does not yet measure whether the rule worked.

In [6]:
usage = defaultdict(lambda: {"predictions": [], "confidences": Counter()})
for row in rule_rows:
    rule = usage[row["rule_id"]]
    if row["predicted_percentile"] is not None:
        rule["predictions"].append(float(row["predicted_percentile"]))
    rule["confidences"][row.get("confidence") or "unknown"] += 1

usage_rows = []
for rule_id, values in usage.items():
    predictions = values["predictions"]
    usage_rows.append({
        "rule_id": rule_id,
        "events": len(predictions),
        "average_prediction": round(mean(predictions), 4) if predictions else None,
        "confidence_counts": dict(values["confidences"]),
    })

usage_rows.sort(key=lambda row: (-row["events"], row["rule_id"]))
display_table(usage_rows, limit=200)

rule_id,events,average_prediction,confidence_counts
Q3-CAL-01,605,0.3679,"{'high': 310, 'medium': 295}"
Q3-CAL-02,481,0.1799,"{'medium': 275, 'high': 206}"
GLB-GUID-01,296,0.1165,"{'high': 140, 'medium': 156}"
GLB-QUAL-01,175,0.1904,"{'medium': 91, 'high': 84}"
Q3-CAL-03,152,0.3852,"{'high': 73, 'medium': 79}"
GLB-EXPECT-03,143,0.8703,"{'high': 112, 'medium': 31}"
GLB-TONE-01,140,0.1827,"{'medium': 79, 'high': 61}"
GLB-QUAL-02,103,0.0881,"{'high': 68, 'medium': 35}"
P5,91,0.2386,"{'medium': 61, 'high': 30}"
GLB-CAP-01,79,0.249,"{'medium': 49, 'high': 30}"


In [12]:
import csv
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

output_path = REPO_ROOT / "knowledge" / "data" / "rule_summary.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

fieldnames = [
    "rule_id",
    "events",
    "average_prediction",
    "confidence_counts",
]

with output_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(usage_rows)

print(f"Saved {len(usage_rows)} rule summaries to {output_path}")

Saved 25 rule summaries to C:\Users\wfpin\Desktop\Projects\explaining-markets\knowledge\data\rule_summary.csv


## 6. Rule outcome summary

This section becomes useful after another job fills `realized_abnormal` and `realized_percentile`. Until then, it reports that no realized rows are available.

In [7]:
outcomes = defaultdict(lambda: {
    "predictions": [],
    "realized_abnormal": [],
    "realized_percentile": [],
})
for row in rule_rows:
    if row["realized_abnormal"] is None and row["realized_percentile"] is None:
        continue
    values = outcomes[row["rule_id"]]
    if row["predicted_percentile"] is not None:
        values["predictions"].append(float(row["predicted_percentile"]))
    if row["realized_abnormal"] is not None:
        values["realized_abnormal"].append(float(row["realized_abnormal"]))
    if row["realized_percentile"] is not None:
        values["realized_percentile"].append(float(row["realized_percentile"]))

outcome_rows = []
for rule_id, values in outcomes.items():
    outcome_rows.append({
        "rule_id": rule_id,
        "realized_events": max(len(values["realized_abnormal"]), len(values["realized_percentile"])),
        "average_prediction": round(mean(values["predictions"]), 4) if values["predictions"] else None,
        "average_realized_abnormal": round(mean(values["realized_abnormal"]), 6) if values["realized_abnormal"] else None,
        "average_realized_percentile": round(mean(values["realized_percentile"]), 4) if values["realized_percentile"] else None,
    })

outcome_rows.sort(key=lambda row: (-row["realized_events"], row["rule_id"]))
display_table(outcome_rows, limit=200)

No rows to display.
